# ANZSIC Company Classifier

Classifies company names to 4-digit ANZSIC codes using a **hierarchical LLM approach**:

1. **Step 1 — Division** — LLM picks from 19 ANZSIC divisions (e.g. "Retail Trade")
2. **Step 2 — Class** — LLM picks the specific 4-digit class within that division (e.g. 4111 "Supermarket and Grocery Stores")

Uses Phi-3-mini (Q4 quantized, CPU-only) for its world knowledge of Australian companies.
Embeddings can't solve this — "Coles" has no semantic overlap with "Supermarket and Grocery Stores".

## Setup

In [ ]:
import pandas as pd
import numpy as np
import re
from llama_cpp import Llama
from huggingface_hub import hf_hub_download
from pathlib import Path

print('Setup OK')

## Download Phi-3-mini GGUF Model

Downloads the Q4_K_M quantized model (~2.2 GB) on first run. Subsequent runs skip the download.

In [ ]:
MODEL_DIR = Path('models')
MODEL_DIR.mkdir(exist_ok=True)

GGUF_REPO = 'microsoft/Phi-3-mini-4k-instruct-gguf'
GGUF_FILE = 'Phi-3-mini-4k-instruct-q4.gguf'
GGUF_PATH = MODEL_DIR / GGUF_FILE

if not GGUF_PATH.exists():
    print(f'Downloading {GGUF_FILE} (~2.2 GB)...')
    hf_hub_download(
        repo_id=GGUF_REPO,
        filename=GGUF_FILE,
        local_dir=str(MODEL_DIR),
    )
    print('Download complete.')
else:
    print(f'Model already downloaded: {GGUF_PATH}')

## Load ANZSIC Reference Data

In [ ]:
anzsic = pd.read_parquet('ANZSIC/ANZSIC.parquet')

# Division lookup: code -> title
divisions = (
    anzsic[anzsic['level'] == 'division']
    .set_index('code')['title']
    .to_dict()
)

# Class-level codes (4-digit) — these are the classification targets
anzsic_classes = (
    anzsic[anzsic['level'] == 'class'][['code', 'title', 'division_code']]
    .reset_index(drop=True)
)

# Classes grouped by division for Step 2 prompts
classes_by_division = {}
for div_code in divisions:
    div_classes = anzsic_classes[anzsic_classes['division_code'] == div_code]
    classes_by_division[div_code] = list(zip(div_classes['code'], div_classes['title']))

print(f'Divisions: {len(divisions)}')
print(f'Total classes: {len(anzsic_classes)}')
print(f'\nClasses per division:')
for code, title in divisions.items():
    print(f'  {code}: {title} ({len(classes_by_division[code])} classes)')

## Load LLM

In [ ]:
llm = Llama(
    model_path=str(GGUF_PATH),
    n_ctx=2048,
    n_threads=4,
    n_gpu_layers=0,
    verbose=False,
)
print('LLM loaded.')

## Classification Functions

Hierarchical two-step classification:
1. LLM picks from 19 ANZSIC divisions
2. LLM picks the specific class within the chosen division

In [ ]:
def classify_division(company_name: str) -> str | None:
    """Step 1: Classify a company name into one of 19 ANZSIC divisions.
    Returns the division code letter, or None if parsing fails."""

    div_list = '\n'.join(
        f'  {code}: {title}' for code, title in sorted(divisions.items())
    )

    prompt = (
        'You are an Australian industry classification expert. '
        'Given a company name, select the ANZSIC division that best describes '
        'the primary industry this company operates in.\n\n'
        f'Company: {company_name}\n\n'
        f'ANZSIC Divisions:\n{div_list}\n\n'
        'Respond with ONLY the single division letter (A-S). No explanation.'
    )

    response = llm(prompt, max_tokens=5, temperature=0.0, stop=['\n'])
    text = response['choices'][0]['text'].strip().upper()

    # Extract single letter
    match = re.search(r'\b([A-S])\b', text)
    if match and match.group(1) in divisions:
        return match.group(1)
    return None


def classify_class(company_name: str, division_code: str) -> tuple[str, str] | None:
    """Step 2: Classify a company name into a specific 4-digit ANZSIC class
    within the given division. Returns (code, title) or None if parsing fails."""

    div_title = divisions[division_code]
    class_list = classes_by_division[division_code]
    valid_codes = {code for code, _ in class_list}

    options = '\n'.join(f'  {code}: {title}' for code, title in class_list)

    prompt = (
        'You are an Australian industry classification expert. '
        f'The company "{company_name}" belongs to ANZSIC Division {division_code}: {div_title}.\n\n'
        'Select the specific ANZSIC class code that best describes this company\'s primary activity.\n\n'
        f'ANZSIC classes in Division {division_code}:\n{options}\n\n'
        'Respond with ONLY the 4-digit ANZSIC code. No explanation.'
    )

    response = llm(prompt, max_tokens=10, temperature=0.0, stop=['\n'])
    text = response['choices'][0]['text'].strip()

    # Extract 4-digit code
    match = re.search(r'\b(\d{4})\b', text)
    if match and match.group(1) in valid_codes:
        code = match.group(1)
        title = dict(class_list)[code]
        return code, title
    return None


def classify_company_to_anzsic(company_name: str) -> dict:
    """Classify a company name to a 4-digit ANZSIC class code using
    hierarchical LLM classification (division -> class)."""

    # Step 1: Division
    div_code = classify_division(company_name)

    if div_code is None:
        # Retry once
        div_code = classify_division(company_name)
        if div_code is None:
            return {
                'anzsic_code': None,
                'anzsic_title': None,
                'division_code': None,
                'division_title': None,
            }

    div_title = divisions[div_code]

    # Step 2: Class within division
    result = classify_class(company_name, div_code)

    if result is None:
        # Retry once
        result = classify_class(company_name, div_code)
        if result is None:
            return {
                'anzsic_code': None,
                'anzsic_title': None,
                'division_code': div_code,
                'division_title': div_title,
            }

    class_code, class_title = result
    return {
        'anzsic_code': class_code,
        'anzsic_title': class_title,
        'division_code': div_code,
        'division_title': div_title,
    }

## Batch Classification

Deduplicates company names before calling the LLM to avoid redundant inference.

In [ ]:
def classify_batch(company_names: pd.Series) -> pd.DataFrame:
    """Classify a batch of company names. Deduplicates before LLM calls."""
    unique_names = company_names.dropna().unique()
    print(f'Classifying {len(unique_names)} unique company names...')

    results = {}
    for i, name in enumerate(unique_names):
        results[name] = classify_company_to_anzsic(name)
        r = results[name]
        status = f"{r['anzsic_code']} {r['anzsic_title']}" if r['anzsic_code'] else f"div={r['division_code']} (class failed)"
        print(f'  [{i + 1}/{len(unique_names)}] {name} → {status}')

    # Map back to original series
    empty = {'anzsic_code': None, 'anzsic_title': None, 'division_code': None, 'division_title': None}
    records = [results.get(name, empty) if pd.notna(name) else empty for name in company_names]
    return pd.DataFrame(records)

## Evaluation

Classify employer names from the synthetic dataset and compare against ground truth ANZSIC codes.

In [ ]:
employees = pd.read_csv(
    'Test Data/synthetic_employees.csv',
    dtype={'employer_industry_anzsic': str, 'host_employer_industry_anzsic': str},
)

# Zero-pad ANZSIC codes to 4 digits
employees['employer_industry_anzsic'] = (
    employees['employer_industry_anzsic']
    .apply(lambda x: str(int(float(x))).zfill(4) if pd.notna(x) and x != 'nan' else None)
)

print(f'Employees: {len(employees)} rows')
print(f'Unique employer names: {employees["employer_name"].nunique()}')

In [ ]:
predictions = classify_batch(employees['employer_name'])

employees['predicted_anzsic_code'] = predictions['anzsic_code'].values
employees['predicted_anzsic_title'] = predictions['anzsic_title'].values
employees['predicted_division_code'] = predictions['division_code'].values
employees['predicted_division_title'] = predictions['division_title'].values

## Results

In [ ]:
# Exact match accuracy (4-digit class level)
exact_match = employees['predicted_anzsic_code'] == employees['employer_industry_anzsic']
print(f'Exact match (4-digit):  {exact_match.mean():.1%}  ({exact_match.sum()}/{len(employees)})')

# Group-level accuracy (first 3 digits)
group_match = (
    employees['predicted_anzsic_code'].str[:3] == employees['employer_industry_anzsic'].str[:3]
)
print(f'Group match (3-digit):  {group_match.mean():.1%}  ({group_match.sum()}/{len(employees)})')

# Division-level accuracy
code_to_div = anzsic_classes.set_index('code')['division_code'].to_dict()
true_div = employees['employer_industry_anzsic'].map(code_to_div)
div_match = employees['predicted_division_code'] == true_div
print(f'Division match:         {div_match.mean():.1%}  ({div_match.sum()}/{len(employees)})')

In [ ]:
# Per-company results (deduplicated view)
company_results = (
    employees[['employer_name', 'employer_industry_anzsic', 'employer_industry_name',
               'predicted_anzsic_code', 'predicted_anzsic_title',
               'predicted_division_code', 'predicted_division_title']]
    .drop_duplicates(subset='employer_name')
    .sort_values('employer_name')
    .reset_index(drop=True)
)

company_results['correct'] = company_results['predicted_anzsic_code'] == company_results['employer_industry_anzsic']

print(f'Per-company accuracy: {company_results["correct"].mean():.1%} ({company_results["correct"].sum()}/{len(company_results)})')
company_results

In [ ]:
# Incorrect predictions
misses = company_results[~company_results['correct']][
    ['employer_name', 'employer_industry_anzsic', 'employer_industry_name',
     'predicted_anzsic_code', 'predicted_anzsic_title',
     'predicted_division_code', 'predicted_division_title']
]
print(f'Incorrect predictions: {len(misses)}')
misses

## Ad-hoc Testing

Test with individual company names.

In [ ]:
test_companies = ['Woolworths', 'BHP', 'Telstra', 'Commonwealth Bank', 'Qantas', 'Coles']

for name in test_companies:
    result = classify_company_to_anzsic(name)
    print(
        f'{name:25s} → '
        f'Div {result["division_code"]}: {result["division_title"]} → '
        f'{result["anzsic_code"]} {result["anzsic_title"]}'
    )